# NB3 — The outcome panel
### The Peltzman Protocol · MSR 2027

Everything so far established **when** each project adopted **what**. This builds the other half: a repo-month panel of what humans did, for all ~45,000 repositories across 72 months.

Source is ClickHouse's public GitHub mirror — free, no account, and a full-month aggregate returns in seconds. The window ends **2025-06** because GitHub trimmed the public event payload around August 2025 and `additions`, `merged` and `merged_by` vanish after it.

| | Outcome | Column |
|---|---|---|
| O1 | Reviewers per PR | distinct actors emitting review events |
| O2 | Review comments per PR | review-comment events |
| O4 | Change size | `additions + deletions` on merged PRs |
| O5 | Self-merge rate | `merged_by` equals the PR's creator |
| O6 | Direct pushes to the default branch | `PushEvent` on `main`/`master` |
| O3 | Time to first review | a second pass — it needs PR-level linkage, not a monthly count |

**One definition matters more than it looks.** A merge is `merged = 1`, never `action = 'merged'`. Before the payload trim the loader records merges the first way; after it, the second. Mixing them would plant a step change in the outcome series at exactly the wrong moment.

**This is a free public endpoint.** One request at a time, a pause between, retry rather than hammer.


## 0 · Setup

In [ ]:
#@title Configuration { display-mode: "form" }
WINDOW_START = "2019-07"  #@param {type:"string"}
WINDOW_END   = "2025-06"  #@param {type:"string"}
REPOS_PER_Q  = 6000       #@param {type:"integer"}
SLEEP_S      = 1.0        #@param {type:"number"}
RETRIES      = 4          #@param {type:"integer"}
CH_HOST      = "https://sql-clickhouse.clickhouse.com/"  #@param {type:"string"}
CH_USER      = "demo"     #@param {type:"string"}

import os, sys, json, time, re, io as _io, subprocess
from pathlib import Path
IN_COLAB=False
try:
    from google.colab import drive
    IN_COLAB=True
except ImportError: pass
DRIVE_OK,DRIVE_ERR=False,None
if IN_COLAB:
    try: drive.mount("/content/drive",force_remount=False); DRIVE_OK=True
    except Exception as e:
        DRIVE_ERR=f"{type(e).__name__}: {e}"; print("!! Drive mount FAILED:",DRIVE_ERR)
DRIVE=("/content/drive/MyDrive/peltzman" if DRIVE_OK else
       "/content/peltzman_LOCAL" if IN_COLAB else os.path.expanduser("~/peltzman_out"))
if IN_COLAB and not DRIVE_OK: print("!! No Drive — state on the runtime disk, destroyed on reset.")
STATE=Path(DRIVE,"state"); CODE=Path("/content" if IN_COLAB else "/tmp")
for _p in (STATE,CODE): Path(_p).mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(CODE))
print("state:",STATE)

In [ ]:
%%writefile /content/peltzman_io.py
"""
Peltzman Protocol - durable, resumable job state.

Guarantee
---------
Kill the process, lose the runtime, close the browser, at ANY instant:
  * no file is ever left half-written,
  * no completed work is ever lost,
  * no work is ever silently duplicated,
  * restarting the cell resumes from the last completed batch.

How
---
The key list is fixed and ordered at job creation, and batch `i` is defined as
a deterministic slice of it. A batch's output is therefore a pure function of
its index, so re-running a batch overwrites its own file and can never
duplicate rows. Completion is represented by the *existence* of that file and
nothing else - there is no separate done-list that can drift out of sync with
the data.

Every file is written to a temporary name, flushed, fsync'd, and then moved
into place with os.replace(), which is atomic. A crash mid-write leaves a
stray .tmp file (harmless, cleaned on next run) and the previous state intact.

Because completion is file existence, the crash window is exactly one batch.
Size batches so one takes a few minutes.
"""
from __future__ import annotations
import json, os, time, hashlib, tempfile, shutil, glob, sys
from pathlib import Path

__all__ = ["Job", "atomic_write_text", "atomic_write_bytes"]


# --------------------------------------------------------------- atomic io --
def atomic_write_bytes(path, data: bytes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())          # no-op on some mounts; harmless
        os.replace(tmp, path)             # atomic within a filesystem
    except BaseException:
        try: os.unlink(tmp)
        except OSError: pass
        raise


def atomic_write_text(path, text: str):
    atomic_write_bytes(path, text.encode("utf-8"))


def _hash_params(params) -> str:
    blob = json.dumps(params, sort_keys=True, default=str).encode()
    return hashlib.sha1(blob).hexdigest()[:10]


# --------------------------------------------------------------------- job --
class Job:
    """A resumable pass over an ordered list of keys.

    >>> job = Job("nb1_workflows", keys=repos, root=DRIVE, batch_size=25,
    ...           params={"detector": "v3"})
    >>> for i, keys in job.pending():
    ...     job.commit(i, [row for k in keys for row in scan(k)])
    >>> df = job.frame()

    Changing `params` starts a new, separate run directory: results computed
    under different settings are never silently mixed.
    """

    def __init__(self, name, keys, root, batch_size=25, params=None, verbose=True):
        self.name = name
        self.params = dict(params or {})
        self.batch_size = int(batch_size)
        self.verbose = verbose
        # NOTE: the key count is deliberately NOT part of the run identity.
        # If the input list changes at all we want a loud error, not a silent
        # fresh run that looks like "nothing was ever done".
        self.dir = Path(root) / name / _hash_params(
            {"params": self.params, "batch_size": self.batch_size})
        self.shards = self.dir / "shards"
        self.shards.mkdir(parents=True, exist_ok=True)

        keyfile = self.dir / "keys.json"
        if keyfile.exists():
            stored = json.loads(keyfile.read_text())
            if stored != list(keys):
                raise ValueError(
                    f"{self.name}: key list changed since this run started "
                    f"({len(stored)} -> {len(keys)}). Change `params` to start "
                    f"a new run, or restore the original list.")
        else:
            atomic_write_text(keyfile, json.dumps(list(keys)))
            atomic_write_text(self.dir / "meta.json", json.dumps({
                "name": name, "created": time.time(), "n_keys": len(keys),
                "batch_size": self.batch_size, "params": self.params,
                "python": sys.version.split()[0]}, indent=1))

        self.keys = list(keys)
        self._sweep_tmp()
        if verbose:
            d, n = self.n_done_batches(), self.n_batches()
            print(f"[{name}] {self.dir}")
            print(f"[{name}] {d}/{n} batches complete"
                  + (f" — resuming at batch {d}" if 0 < d < n else
                     " — nothing to do" if d == n else " — starting fresh"))

    # ---------------------------------------------------------- internals --
    def _sweep_tmp(self):
        """Remove leftovers from a crash mid-write. They are never valid."""
        for t in glob.glob(str(self.shards / "*.tmp")) + glob.glob(str(self.dir / "*.tmp")):
            try: os.unlink(t)
            except OSError: pass

    def _path(self, i) -> Path:
        return self.shards / f"batch-{i:06d}.jsonl"

    def n_batches(self) -> int:
        return (len(self.keys) + self.batch_size - 1) // self.batch_size

    def batch_keys(self, i):
        return self.keys[i * self.batch_size:(i + 1) * self.batch_size]

    def done_batches(self):
        return {i for i in range(self.n_batches()) if self._path(i).exists()}

    def n_done_batches(self) -> int:
        return len(self.done_batches())

    # ------------------------------------------------------------- driving --
    def pending(self):
        """Yield (batch_index, keys) for every batch not yet committed."""
        done = self.done_batches()
        for i in range(self.n_batches()):
            if i not in done:
                yield i, self.batch_keys(i)

    def commit(self, i, rows):
        """Atomically record this batch's output. Idempotent by construction."""
        payload = "".join(json.dumps(r, separators=(",", ":"), default=str) + "\n"
                          for r in rows)
        atomic_write_text(self._path(i), payload)
        self.beat(i, len(rows))

    def beat(self, i, n_rows=None, note=None):
        atomic_write_text(self.dir / "heartbeat.json", json.dumps({
            "at": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "last_batch": i, "rows": n_rows, "note": note,
            "done": self.n_done_batches(), "total": self.n_batches()}, indent=1))

    def run(self, fn, on_error="record", progress_every=1, after_batch=None):
        """Drive the whole job. `fn(key) -> list[dict]`.

        on_error='record'  capture the failure as a row and move on (default)
                 'skip'    drop the key silently
                 'raise'   stop, keeping every committed batch
        after_batch(job, i)  optional hook run after each successful commit,
                 e.g. to replicate state off the runtime. A failing hook never
                 loses the batch - it is already on disk before the hook runs.
        """
        t0, n_before = time.time(), self.n_done_batches()
        for i, keys in self.pending():
            rows = []
            for k in keys:
                try:
                    rows.extend(fn(k) or [])
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    if on_error == "raise":
                        raise
                    if on_error == "record":
                        rows.append({"_key": k, "_error": f"{type(e).__name__}: {e}"})
            self.commit(i, rows)
            if after_batch is not None:
                try: after_batch(self, i)
                except Exception as e:
                    print(f"  after_batch hook failed ({type(e).__name__}: {e}) — "
                          f"batch is still committed locally")
            done = self.n_done_batches()
            if self.verbose and (done - n_before) % progress_every == 0:
                el = time.time() - t0
                per = el / max(1, done - n_before)
                left = (self.n_batches() - done) * per
                print(f"  batch {done}/{self.n_batches()}  "
                      f"{per:5.1f}s/batch  ~{left/60:5.1f} min remaining")
        if self.verbose:
            print(f"[{self.name}] complete: {self.n_batches()} batches")

    # ------------------------------------------------------------ reading --
    def rows(self, drop_errors=True):
        """Iterate every committed row. Truncated trailing lines - which an
        atomic write cannot produce, but a copied/synced file might - are
        skipped and counted rather than raising."""
        bad = 0
        for i in sorted(self.done_batches()):
            with open(self._path(i), "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        bad += 1
                        continue
                    if drop_errors and isinstance(r, dict) and "_error" in r:
                        continue
                    yield r
        if bad and self.verbose:
            print(f"[{self.name}] warning: skipped {bad} unparseable line(s) — "
                  f"run .verify() to locate")

    def frame(self, drop_errors=True):
        import pandas as pd
        return pd.DataFrame(list(self.rows(drop_errors=drop_errors)))

    def errors(self):
        return [r for r in self.rows(drop_errors=False)
                if isinstance(r, dict) and "_error" in r]

    # -------------------------------------------------------- maintenance --
    def verify(self):
        """Check every committed shard parses. Returns list of bad batch ids."""
        bad = []
        for i in sorted(self.done_batches()):
            try:
                for line in open(self._path(i), encoding="utf-8"):
                    if line.strip():
                        json.loads(line)
            except Exception as e:
                bad.append((i, f"{type(e).__name__}: {e}"))
        if self.verbose:
            print(f"[{self.name}] verify: {self.n_done_batches()} shards, "
                  f"{len(bad)} damaged")
        return bad

    def reset(self, batches=None):
        """Delete shards so they are recomputed. `batches=None` resets damaged
        ones only; pass an iterable of indices, or 'all'."""
        if batches == "all":
            targets = sorted(self.done_batches())
        elif batches is None:
            targets = [i for i, _ in self.verify()]
        else:
            targets = list(batches)
        for i in targets:
            try: os.unlink(self._path(i))
            except OSError: pass
        if self.verbose:
            print(f"[{self.name}] reset {len(targets)} batch(es)")
        return targets

    def retry_errors(self):
        """Re-queue every batch that recorded at least one failure."""
        hit = set()
        for i in sorted(self.done_batches()):
            for line in open(self._path(i), encoding="utf-8"):
                if '"_error"' in line:
                    hit.add(i); break
        return self.reset(hit)

    def export(self, path, fmt="parquet"):
        """Write the whole result set as one file, atomically."""
        df = self.frame()
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_suffix(p.suffix + ".tmp")
        if fmt == "parquet":
            df.to_parquet(tmp, index=False)
        else:
            df.to_csv(tmp, index=False)
        os.replace(tmp, p)
        if self.verbose:
            print(f"[{self.name}] exported {len(df):,} rows -> {p}")
        return p

    def status(self):
        hb = self.dir / "heartbeat.json"
        return {"dir": str(self.dir), "batches": self.n_batches(),
                "done": self.n_done_batches(), "keys": len(self.keys),
                "heartbeat": json.loads(hb.read_text()) if hb.exists() else None}

In [ ]:
%%writefile /content/test_peltzman_io.py
"""Crash-safety tests. These simulate the failure modes Colab actually has:
killed mid-batch, killed mid-write, re-run after partial completion,
corrupted shard on disk, and changing inputs underneath a running job."""
import sys, os, json, shutil, tempfile, subprocess, textwrap
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from peltzman_io import Job, atomic_write_text

fails = []
def check(name, got, want):
    if got != want: fails.append(f"{name}\n    got  {got!r}\n    want {want!r}")
def ok(name, cond):
    if not cond: fails.append(name)

ROOT = tempfile.mkdtemp()
KEYS = [f"repo{i:03d}" for i in range(100)]
def work(k): return [{"key": k, "v": int(k[-3:])}]

# --- 1. clean run ---------------------------------------------------------
j = Job("t1", KEYS, ROOT, batch_size=10, verbose=False)
check("batch count", j.n_batches(), 10)
j.run(work)
check("all committed", j.n_done_batches(), 10)
check("row count", len(list(j.rows())), 100)

# --- 2. resume after interruption ----------------------------------------
j2 = Job("t2", KEYS, ROOT, batch_size=10, verbose=False)
processed = []
class Boom(Exception): pass
def flaky(k):
    processed.append(k)
    if k == "repo035": raise Boom()
    return work(k)
try:
    j2.run(flaky, on_error="raise")
except Boom:
    pass
check("3 batches survived the crash", j2.n_done_batches(), 3)
check("rows kept", len(list(j2.rows())), 30)

before = len(processed)
j3 = Job("t2", KEYS, ROOT, batch_size=10, verbose=False)   # fresh object = new session
check("resume sees prior work", j3.n_done_batches(), 3)
pend = [i for i, _ in j3.pending()]
check("resumes at batch 3", pend[0], 3)
j3.run(work)
check("completes", j3.n_done_batches(), 10)
check("no duplicated rows", len(list(j3.rows())), 100)
check("keys unique", len({r["key"] for r in j3.rows()}), 100)
check("did not redo committed batches", processed[:before] == processed[:before], True)
redone = [k for k in KEYS[:30]]
check("first 30 keys not reprocessed after resume",
      sum(1 for k in redone if k in processed[before:]), 0)

# --- 3. killed mid-write leaves no corruption ----------------------------
# Run a real subprocess and SIGKILL it while it writes, then reopen.
script = textwrap.dedent(f"""
    import sys, time, os
    sys.path.insert(0, {os.path.dirname(os.path.abspath(__file__))!r})
    from peltzman_io import Job
    KEYS = [f"repo{{i:03d}}" for i in range(100)]
    j = Job("t3", KEYS, {ROOT!r}, batch_size=10, verbose=False)
    def slow(k):
        time.sleep(0.05)
        return [{{"key": k}}]
    j.run(slow)
""")
sp = os.path.join(ROOT, "kill_me.py"); open(sp, "w").write(script)
p = subprocess.Popen([sys.executable, sp])
import time as _t; _t.sleep(1.6)
p.kill(); p.wait()

j4 = Job("t3", KEYS, ROOT, batch_size=10, verbose=False)
n_after_kill = j4.n_done_batches()
ok("SIGKILL mid-run committed some batches", 0 < n_after_kill < 10)
check("no damaged shards after SIGKILL", j4.verify(), [])
tmps = [f for f in os.listdir(j4.shards) if f.endswith(".tmp")]
check("stray .tmp files swept on reopen", tmps, [])
rows_after_kill = list(j4.rows())
check("rows intact and whole", len(rows_after_kill), n_after_kill * 10)
j4.run(lambda k: [{"key": k}])
check("finishes after kill", j4.n_done_batches(), 10)
check("still exactly 100 rows", len({r["key"] for r in j4.rows()}), 100)

# --- 4. externally corrupted shard is detected and repairable ------------
j5 = Job("t5", KEYS, ROOT, batch_size=10, verbose=False); j5.run(work)
victim = j5._path(4)
open(victim, "a").write('{"key": "trunc", "v": ')      # half a line
bad = j5.verify()
check("damage detected", [i for i, _ in bad], [4])
check("rows() tolerates it rather than raising", len(list(j5.rows())), 100)
j5.reset()                                              # repairs damaged only
check("damaged shard requeued", j5.n_done_batches(), 9)
j5.run(work)
check("repaired to full", len(list(j5.rows())), 100)

# --- 5. errors are captured, not fatal, and retryable --------------------
j6 = Job("t6", KEYS, ROOT, batch_size=10, verbose=False)
state = {"fail": True}
def sometimes(k):
    if state["fail"] and k == "repo042": raise RuntimeError("rate limited")
    return work(k)
j6.run(sometimes)
check("job completed despite failure", j6.n_done_batches(), 10)
check("one error recorded", len(j6.errors()), 1)
check("good rows still 99", len(list(j6.rows())), 99)
state["fail"] = False
j6.retry_errors()
j6.run(sometimes)
check("after retry, 100 rows", len(list(j6.rows())), 100)
check("no errors remain", len(j6.errors()), 0)

# --- 6. changed inputs cannot silently corrupt a run ---------------------
try:
    Job("t6", KEYS[:50], ROOT, batch_size=10, verbose=False)
    fails.append("changing the key list should have raised")
except ValueError:
    pass
j7 = Job("t6", KEYS, ROOT, batch_size=10, params={"detector": "v2"}, verbose=False)
check("different params = separate run", j7.n_done_batches(), 0)

# --- 7. atomic_write_text never leaves a partial file --------------------
target = os.path.join(ROOT, "atomic.json")
atomic_write_text(target, json.dumps({"a": 1}))
try:
    atomic_write_text(target, None)      # TypeError inside
except Exception:
    pass
check("original survives a failed write", json.load(open(target)), {"a": 1})
check("no .tmp residue", [f for f in os.listdir(ROOT) if f.startswith("atomic.json.")], [])

shutil.rmtree(ROOT, ignore_errors=True)
print(f"ran {7} scenarios / {28} assertions")
if fails:
    print(f"\nFAILURES ({len(fails)}):\n" + "\n\n".join(fails)); sys.exit(1)
print("all passed — crash safety verified")

In [ ]:
import importlib, peltzman_io; importlib.reload(peltzman_io)
from peltzman_io import Job, atomic_write_text
r=subprocess.run([sys.executable,str(CODE/"test_peltzman_io.py")],capture_output=True,text=True)
print(r.stdout.strip().splitlines()[-1] if r.stdout else "NO OUTPUT")
if r.returncode: print(r.stderr[-1500:]); raise SystemExit("checkpoint tests FAILED")

## 1 · The repository universe and the endpoint

Treated and never-treated together — the panel has to cover both arms, and the never-treated repos are what identify the effect.

In [ ]:
import pandas as pd, requests
tre  = pd.read_parquet(Path(STATE,"nb1_treatments.parquet"))
never= json.loads(Path(STATE,"nb1_controls.json").read_text())
treated = sorted(set(tre[tre.treatment=="A_first_gating"].repo))
SAFE = re.compile(r"^[A-Za-z0-9._-]+/[A-Za-z0-9._-]+$")
universe = sorted({r for r in (treated + never) if SAFE.match(r)})
print(f"treated {len(treated):,} · never-treated {len(never):,} · universe {len(universe):,}")
dropped = (len(treated)+len(never)) - len(universe)
if dropped: print(f"  {dropped} names dropped as unsafe for a SQL literal")

def ch(sql, timeout=300, retries=RETRIES):
    last=None
    for a in range(retries):
        try:
            r=requests.post(CH_HOST, params={"user":CH_USER},
                            data=f"{sql} FORMAT TSVWithNames".encode(), timeout=timeout)
            if r.status_code==200: return r.text
            last=f"HTTP {r.status_code}: {r.text[:200]}"
        except Exception as e:
            last=f"{type(e).__name__}: {e}"
        time.sleep(2**a)
    raise RuntimeError(last)

print("\nendpoint check:", ch("SELECT count() AS n FROM github.events WHERE created_at >= '2022-06-01' AND created_at < '2022-06-02'").strip())

### Which timestamp is the event time?

The sibling deployment stores the *object's* timestamp in `created_at`, which would silently mis-bin every observation. Verify on this one before building anything on it.

In [ ]:
chk = ch("""
SELECT toYYYYMM(created_at) AS by_created, count() AS n
FROM github.events
WHERE created_at >= '2022-06-01' AND created_at < '2022-07-01'
  AND event_type = 'PullRequestEvent'
GROUP BY by_created ORDER BY n DESC LIMIT 5
""")
print(chk)
print("Expect ONE month (202206) with essentially all rows. Several months means\n"
      "created_at is the object timestamp and the queries below must use file_time.")

## 2 · Extract the panel

One query per (month, repo-chunk). Chunking keeps the `IN` list inside ClickHouse's query-size limit and keeps each request small enough to be a good citizen.

In [ ]:
def months(a, b):
    y, m = int(a[:4]), int(a[5:7]); out=[]
    while f"{y:04d}-{m:02d}" <= b:
        out.append(f"{y:04d}-{m:02d}")
        m += 1
        if m == 13: y, m = y+1, 1
    return out

MONTHS = months(WINDOW_START, WINDOW_END)
CHUNKS = [universe[i:i+REPOS_PER_Q] for i in range(0, len(universe), REPOS_PER_Q)]
KEYS   = [f"{m}|{i}" for m in MONTHS for i in range(len(CHUNKS))]
print(f"{len(MONTHS)} months × {len(CHUNKS)} chunks = {len(KEYS)} queries")

def month_bounds(m):
    y, mo = int(m[:4]), int(m[5:7])
    ny, nmo = (y+1, 1) if mo == 12 else (y, mo+1)
    return f"{y:04d}-{mo:02d}-01", f"{ny:04d}-{nmo:02d}-01"

PANEL_SQL = """
SELECT repo_name AS repo,
       countIf(event_type='PullRequestEvent' AND action='opened')            AS prs_opened,
       countIf(event_type='PullRequestEvent' AND merged=1)                   AS prs_merged,
       countIf(event_type='PullRequestReviewEvent')                          AS reviews,
       uniqExactIf(actor_login, event_type='PullRequestReviewEvent')         AS reviewers,
       uniqExactIf(number, event_type='PullRequestReviewEvent')              AS prs_reviewed,
       countIf(event_type='PullRequestReviewCommentEvent')                   AS review_comments,
       countIf(event_type='PullRequestReviewEvent' AND review_state='approved')          AS approvals,
       countIf(event_type='PullRequestReviewEvent' AND review_state='changes_requested') AS changes_req,
       sumIf(additions+deletions, event_type='PullRequestEvent' AND merged=1)            AS churn_merged,
       sumIf(changed_files,       event_type='PullRequestEvent' AND merged=1)            AS files_merged,
       countIf(event_type='PullRequestEvent' AND merged=1 AND merged_by!=''
               AND creator_user_login!='')                                   AS merged_attributable,
       countIf(event_type='PullRequestEvent' AND merged=1 AND merged_by!=''
               AND merged_by=creator_user_login)                             AS self_merged,
       countIf(event_type='PushEvent')                                       AS pushes,
       countIf(event_type='PushEvent' AND ref IN ('refs/heads/main','refs/heads/master')) AS pushes_default,
       uniqExactIf(actor_login, event_type='PullRequestEvent' AND action='opened')        AS pr_authors
FROM github.events
WHERE created_at >= '{lo}' AND created_at < '{hi}'
  AND event_type IN ('PullRequestEvent','PullRequestReviewEvent',
                     'PullRequestReviewCommentEvent','PushEvent')
  AND repo_name IN ({repos})
GROUP BY repo_name
"""

def fetch(key):
    m, ci = key.split("|"); lo, hi = month_bounds(m)
    repos = ",".join("'" + r + "'" for r in CHUNKS[int(ci)])
    txt = ch(PANEL_SQL.format(lo=lo, hi=hi, repos=repos))
    lines = txt.rstrip("\n").split("\n")
    if len(lines) < 2: return []
    hdr = lines[0].split("\t")
    out = []
    for ln in lines[1:]:
        v = ln.split("\t")
        if len(v) != len(hdr): continue
        row = {"month": m}
        for k, x in zip(hdr, v):
            row[k] = x if k == "repo" else int(x)
        out.append(row)
    time.sleep(SLEEP_S)
    return out

panel = Job("nb3_panel", KEYS, STATE, batch_size=1,
            params={"window":[WINDOW_START,WINDOW_END], "chunk":REPOS_PER_Q,
                    "n_repos":len(universe), "host":CH_HOST})
panel.run(fetch, progress_every=10)
print(f"\nrows {sum(1 for _ in panel.rows()):,} · failures {len(panel.errors())}")
for e in panel.errors()[:5]: print("  ", str(e)[:160])

## 3 · Sanity checks before anything is believed

A panel that looks plausible in aggregate can still be wrong per-repo. These three checks catch the failures that would otherwise surface as a mysterious result in NB5.

In [ ]:
df = pd.DataFrame(panel.rows())
print(f"panel: {len(df):,} repo-months · {df.repo.nunique():,} repos · {df.month.nunique()} months")

cov = df.groupby("month").agg(repos=("repo","nunique"), prs=("prs_opened","sum"))
print("\nmonthly coverage (every 6th month)")
print(cov.iloc[::6].to_string())

print("\nzero-activity months:", f"{(df.prs_opened==0).mean():.1%} of repo-months have no PR opened")
print("repos never appearing:", f"{len(set(universe))-df.repo.nunique():,} of {len(universe):,}")

print("\nfield population on merged PRs (should be high, pre-trim window)")
mg = df.prs_merged.sum()
for c in ("churn_merged","files_merged","merged_attributable","self_merged"):
    print(f"  {c:22s} {df[c].sum():>14,}   ({df[c].sum()/max(1,mg):.2f} per merged PR)")
print("\nIf merged_attributable is near zero, self-merge (O5) is not measurable and\n"
      "should be dropped rather than reported as a null.")

## 4 · Time to first review (O3)

A monthly count cannot express latency, so this pass links each pull request opened in a month to its first review, allowing up to 60 days for that review to arrive. Run it after the main panel — if the endpoint struggles, the panel above still stands on its own.

In [ ]:
LAT_SQL = """
SELECT repo, quantile(0.5)(lat_h) AS med_lat_h, avg(lat_h) AS mean_lat_h, count() AS n_pr
FROM (
  SELECT o.repo_name AS repo,
         dateDiff('hour', o.t0, r.t1) AS lat_h
  FROM (
    SELECT repo_name, number, min(created_at) AS t0 FROM github.events
    WHERE created_at >= '{lo}' AND created_at < '{hi}'
      AND event_type='PullRequestEvent' AND action='opened'
      AND repo_name IN ({repos}) GROUP BY repo_name, number
  ) AS o
  INNER JOIN (
    SELECT repo_name, number, min(created_at) AS t1 FROM github.events
    WHERE created_at >= '{lo}' AND created_at < '{hi2}'
      AND event_type='PullRequestReviewEvent'
      AND repo_name IN ({repos}) GROUP BY repo_name, number
  ) AS r ON o.repo_name = r.repo_name AND o.number = r.number
  WHERE r.t1 >= o.t0
) GROUP BY repo
"""

def month_plus(m, k):
    y, mo = int(m[:4]), int(m[5:7]); mo += k
    y += (mo-1)//12; mo = (mo-1)%12 + 1
    return f"{y:04d}-{mo:02d}-01"

def fetch_lat(key):
    m, ci = key.split("|"); lo, hi = month_bounds(m)
    repos = ",".join("'" + r + "'" for r in CHUNKS[int(ci)])
    txt = ch(LAT_SQL.format(lo=lo, hi=hi, hi2=month_plus(m,2), repos=repos), timeout=420)
    lines = txt.rstrip("\n").split("\n")
    if len(lines) < 2: return []
    hdr = lines[0].split("\t"); out=[]
    for ln in lines[1:]:
        v = ln.split("\t")
        if len(v)!=len(hdr): continue
        d = dict(zip(hdr, v))
        out.append({"month": m, "repo": d["repo"], "med_lat_h": float(d["med_lat_h"]),
                    "mean_lat_h": float(d["mean_lat_h"]), "n_pr_reviewed": int(d["n_pr"])})
    time.sleep(SLEEP_S)
    return out

lat = Job("nb3_latency", KEYS, STATE, batch_size=1,
          params={"window":[WINDOW_START,WINDOW_END],"chunk":REPOS_PER_Q,"grace_months":2})
lat.run(fetch_lat, progress_every=10)
dl = pd.DataFrame(lat.rows())
print(f"\nlatency rows {len(dl):,} · failures {len(lat.errors())}")
if len(dl): print(dl[["med_lat_h","mean_lat_h","n_pr_reviewed"]].describe().to_string())

## 5 · Export

In [ ]:
if len(dl):
    df = df.merge(dl, on=["repo","month"], how="left")
out = Path(STATE,"nb3_panel.parquet")
tmp = out.with_suffix(".parquet.tmp"); df.to_parquet(tmp, index=False); os.replace(tmp, out)

summary = {"n_rows": len(df), "n_repos": int(df.repo.nunique()), "n_months": int(df.month.nunique()),
           "window":[WINDOW_START,WINDOW_END],
           "totals": {c: int(df[c].sum()) for c in
                      ("prs_opened","prs_merged","reviews","review_comments",
                       "approvals","changes_req","self_merged","merged_attributable",
                       "pushes_default","churn_merged")},
           "panel_failures": len(panel.errors()), "latency_failures": len(lat.errors()),
           "latency_rows": int(len(dl))}
atomic_write_text(Path(STATE,"nb3_summary.json"), json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
print(f"\nwrote {len(df):,} repo-months -> {out}  ({out.stat().st_size/2**20:.0f} MB)")